# Lab: inter-shell links, live latency and Cesium replay
An Earth observation (EO) satellite collects far more data than it can download when it flies over a ground station. A solution is to **relay** its data through a communication constellation, with optical inter-satellite links between both: **inter-shell links**.

In this lab you will:
1. build **EO Relay**, a small IRIS²-like communication shell (Walker Delta, 1 200 km) whose job is to relay EO data, and an **EO constellation of 6 satellites** sharing the same dawn-dusk sun-synchronous orbit (SSO, 550 km)
2. add **two ground stations in Toulouse**: CNES, which only talks to the EO satellites, and ISAE-SUPAERO, which only talks to EO Relay
3. let the `inter-shell-sso-priority` strategy connect every EO satellite to an EO Relay satellite, and **count the inter-shell handovers**
4. write an **event factory** that adds the satellite positions to every ISL event
5. maintain a **graph of the network in real time** from the events, and measure at each tick the **propagation latency** from both ground stations to every EO satellite
6. replay everything in an **animated Cesium page**, latencies and handover counter included

This lab builds on the event bus of [optical_terminal_lab.ipynb](optical_terminal_lab.ipynb).

In [ ]:
import math
from concurrent.futures import ThreadPoolExecutor

import networkx as nx

from sat_com_application.event_bus.bus import EventBus
from sat_com_application.event_bus.events import (
    EventType,
    ISLCreatedEvent,
    ISLUpdatedEvent,
    ISLRemovedEvent,
    GSLCreatedEvent,
    GSLRemovedEvent,
)
from sat_com_application.event_bus.factories import DefaultISLEventFactory, EventFactories
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty
from sat_com_model.models import InterSatelliteLink

from cesium_graph_visualizer import AnimatedCesiumGraphVisualizer
from dynamic_topology_graph import DynamicTopologyGraph

## 1. Two shells and two ground stations
**The EO Relay shell** is the IRIS²-like shell of the optical lab (10 planes × 10 satellites, 55°, 1 200 km), renamed after its job here, with a **5th terminal** dedicated to inter-shell links. A terminal is *inter-shell capable* when its `orientation_possibilities` contains an `INTER_SHELL_*` value.

**The EO shell** is a `dawn_dusk_sso_shells` entry: 6 satellites spread on the **same** dawn-dusk sun-synchronous orbit at 550 km. The orbit inclination is derived from the altitude, and the plane follows the day/night boundary. EO satellites don't talk to each other: their addressing plan is `EMPTY` (no intra-shell link), and each one carries a single optical terminal, for inter-shell links. (`EMPTY` on an SSO shell needs satComTopology 5.0.4 or later.)

**The strategy** `inter_satellite_link_connection_strategy="inter-shell-sso-priority"` runs in two phases at every tick:
1. inter-shell terminals are assigned first: each free one is linked to the best feasible terminal of the other shell, and an existing inter-shell link is kept as long as it stays feasible
2. the remaining terminals follow the addressing plan of their shell

Inter-shell links are always computed with the `advanced` optical model (checked against both terminals), whatever `optical_inter_satellite_link_model` says. We keep the default `basic` model for the intra-shell links: it is faster, and their distance is all we need.

**The ground stations** are written inline (`ground_objects`), one group each, so each one gets its own connection rules. The **whitelist** of the ground strategy, `shells_whitelist`, restricts the satellites a ground station may connect to:
- **CNES** (Toulouse Space Centre) only connects to `EO SSO`: it downloads the EO data directly, when an EO satellite flies over Toulouse
- **ISAE-SUPAERO** only connects to `EO Relay`: it reaches the EO satellites through the relay and the inter-shell links

Both use `best-angle-until-disconnection` with a single satellite at a time (`maximum_connected_satellites: 1`).

We simulate 2 hours (a bit more than one orbit of both shells), one tick every 30 seconds, **from 05:00 to 07:00 UTC**. A dawn-dusk orbit follows the day/night boundary: it passes near Toulouse around dawn and dusk (local time), so this is when CNES can see the EO satellites. At midnight, CNES would not see a single one.

In [ ]:
def optical_terminal(azimuth, descriptor, orientation_possibilities=None):
    return {
        "azimuth_angle_configuration": azimuth,
        "elevation_angle_configuration": -90.0,
        "descriptor": descriptor,
        "orientation_possibilities": orientation_possibilities or [descriptor],
    }


def toulouse_ground_station(identifier, object_id, label, latitude, longitude, shell, elevation):
    return {
        "identifier": identifier,
        "type": "ground_station",
        "ground_objects": [
            {"id": object_id, "label": label, "latitude": latitude, "longitude": longitude, "altitude": 150.0}
        ],
        "connectivity_properties": {
            "elevation_above_horizon": elevation,
            "shells_whitelist": [shell],
            "maximum_connected_satellites": 1,
            "ground_to_space_connections_strategy": "best-angle-until-disconnection",
        },
    }


INTER_SHELL_TERMINAL = optical_terminal(90.0, "INTER_SHELL_LEFT", ["INTER_SHELL_LEFT", "INTER_SHELL_RIGHT"])

lab_config = {
    "simulation_name": "Inter Shell Lab",
    "start_date": "2026-01-01 05:00:00.000000",
    "end_date": "2026-01-01 07:00:00.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "inter_satellite_link_connection_strategy": "inter-shell-sso-priority",
    "ground_objects_properties": [
        toulouse_ground_station("CNES EO Station", 0, "CNES", 43.5609, 1.4993, shell="EO SSO", elevation=10),
        toulouse_ground_station("ISAE-SUPAERO Gateway", 1, "ISAE-SUPAERO", 43.5654, 1.4743, shell="EO Relay", elevation=20),
    ],
    "walker_shells": [
        {
            "type": "delta",
            "static_addressing_plan": "FULL_MESH_WALKER_DELTA",
            "satellite_property": {
                "optical_inter_satellite_link_installations": [
                    optical_terminal(0.0, "INTRA_PLANE_FORWARD"),
                    optical_terminal(-180.0, "INTRA_PLANE_BACKWARD"),
                    optical_terminal(90.0, "INTER_PLANE_LEFT"),
                    optical_terminal(-90.0, "INTER_PLANE_RIGHT"),
                    INTER_SHELL_TERMINAL,
                ]
            },
            "constellation_property": {
                "identifier": "EO Relay",
                "amount_of_orbit_plane": 10,
                "amount_of_satellite_per_orbit_plane": 10,
                "inclination": 55.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": 13.16,
            },
            "orbital_connectivity_property": {"adjacent_inter_satellite_shifting": 0},
        }
    ],
    "dawn_dusk_sso_shells": [
        {
            "static_addressing_plan": "EMPTY",
            "satellite_property": {"optical_inter_satellite_link_installations": [INTER_SHELL_TERMINAL]},
            "constellation_property": {
                "identifier": "EO SSO",
                "altitude_in_km": 550.0,
                "amount_of_satellite": 6,
            },
            "orbital_connectivity_property": {"adjacent_inter_satellite_shifting": 0},
        }
    ],
}

simulation_properties = SimulationProperty(**lab_config)

## 2. An event factory that adds the satellite positions
To draw the constellation we need the position of every satellite at every tick. The default ISL events only carry satellite ids, so we **customise the events** with an event factory:
1. subclass `ISLCreatedEvent` / `ISLUpdatedEvent` with the extra fields: position (longitude, latitude, altitude in meters), name and shell of both satellites
2. subclass `DefaultISLEventFactory`: let it build the standard event, then extend it
3. inject it through `EventFactories` (the other factories keep their default)

The events keep their `type`, so the subscribers of `EventType.ISL_CREATED` / `ISL_UPDATED` receive them.

Every ISL is either created or updated at each tick, so the ISL events tell us where every linked satellite is, at every tick. EO Relay satellites always have intra-shell links; an EO satellite is only seen through its inter-shell link, so on the rare tick where it has none, Cesium interpolates its position from the neighbouring ticks. This is exactly what an emulator would do: it only sees the events.

In [ ]:
class PositionedISLCreatedEvent(ISLCreatedEvent):
    satellite_a_name: str
    satellite_b_name: str
    satellite_a_shell: str
    satellite_b_shell: str
    satellite_a_position: tuple[float, float, float]  # longitude, latitude, altitude (m)
    satellite_b_position: tuple[float, float, float]


class PositionedISLUpdatedEvent(ISLUpdatedEvent):
    satellite_a_name: str
    satellite_b_name: str
    satellite_a_shell: str
    satellite_b_shell: str
    satellite_a_position: tuple[float, float, float]
    satellite_b_position: tuple[float, float, float]


def position_of(topology_object):
    coordinates = topology_object.get_coordinates()
    return coordinates.longitude, coordinates.latitude, coordinates.altitude


class PositionedISLEventFactory(DefaultISLEventFactory):

    @staticmethod
    def _satellites_information(link: InterSatelliteLink) -> dict:
        satellite_a = link.get_satellite_source()
        satellite_b = link.get_satellite_destination()
        return dict(
            satellite_a_name=satellite_a.satellite_name,
            satellite_b_name=satellite_b.satellite_name,
            satellite_a_shell=satellite_a.shell.identifier,
            satellite_b_shell=satellite_b.shell.identifier,
            satellite_a_position=position_of(satellite_a),
            satellite_b_position=position_of(satellite_b),
        )

    def created(self, link: InterSatelliteLink) -> PositionedISLCreatedEvent:
        event = super().created(link)
        return PositionedISLCreatedEvent(**event.model_dump(), **self._satellites_information(link))

    def updated(self, link: InterSatelliteLink) -> PositionedISLUpdatedEvent:
        event = super().updated(link)
        return PositionedISLUpdatedEvent(**event.model_dump(), **self._satellites_information(link))


event_factories = EventFactories(isl_event_factory=PositionedISLEventFactory())

## 3. Listen to the events
We record, from the events only:
- `satellites`: for every satellite, its name, its shell and its position at every tick (from `ISL_CREATED` and `ISL_UPDATED`)
- `link_lifetimes`: for every pair of objects, the time intervals during which a link existed (opened by `*_CREATED`, closed by `*_REMOVED`), for the ISLs and the ground station links (`GSL_*`)

On the way, we count the **inter-shell handovers**: an EO satellite whose inter-shell link is removed and replaced by a new one in the same tick. The obsolete links are removed before the new ones are created, so the `ISL_REMOVED` event always comes first. `handover_counter` holds the running total, and its value at each handover is kept for Cesium.

In [ ]:
satellites = {}
open_links = {}
link_lifetimes = {}
last_inter_shell_removal = {}
handovers = []
handover_counter = {"total": 0, "samples": []}


def record_positions(event):
    for satellite_id, name, shell, position in (
        (event.satellite_a_id, event.satellite_a_name, event.satellite_a_shell, event.satellite_a_position),
        (event.satellite_b_id, event.satellite_b_name, event.satellite_b_shell, event.satellite_b_position),
    ):
        satellite = satellites.setdefault(satellite_id, {"name": name, "shell": shell, "positions": {}})
        satellite["positions"][event.sim_time] = position


def open_link(link_id, object_a_id, object_b_id, direction, start_time):
    open_links[link_id] = (object_a_id, object_b_id, direction, start_time)


def close_link(link_id, end_time):
    object_a_id, object_b_id, direction, start_time = open_links.pop(link_id)
    pair = tuple(sorted((object_a_id, object_b_id)))
    lifetime = link_lifetimes.setdefault(pair, {"direction": direction, "intervals": []})
    lifetime["intervals"].append((start_time, end_time))
    return object_a_id, object_b_id


def eo_and_relay_ends(event):
    """(EO satellite id, EO Relay satellite name) of an inter-shell link."""
    if event.satellite_a_shell == "EO SSO":
        return event.satellite_a_id, event.satellite_b_name
    return event.satellite_b_id, event.satellite_a_name


async def on_isl_created(event: PositionedISLCreatedEvent):
    record_positions(event)
    open_link(event.link_id, event.satellite_a_id, event.satellite_b_id, event.direction, event.sim_time)

    if event.direction != "inter_shell":
        return
    eo_id, relay_name = eo_and_relay_ends(event)
    removal = last_inter_shell_removal.pop(eo_id, None)
    if removal is not None and removal[0] == event.sim_time:
        handover_counter["total"] += 1
        handover_counter["samples"].append((event.sim_time, handover_counter["total"]))
        handovers.append({"time": event.sim_time, "eo": satellites[eo_id]["name"], "from": removal[1], "to": relay_name})
        print(
            f"[{event.sim_time:%H:%M:%S}] handover #{handover_counter['total']:<3} {satellites[eo_id]['name']}:"
            f" {removal[1]} -> {relay_name} ({event.distance_km:.0f} km)"
        )


async def on_isl_updated(event: PositionedISLUpdatedEvent):
    record_positions(event)


async def on_isl_removed(event: ISLRemovedEvent):
    satellite_a_id, satellite_b_id = close_link(event.link_id, event.sim_time)
    if event.direction == "inter_shell":
        eo_id, relay_id = (
            (satellite_a_id, satellite_b_id) if satellites[satellite_a_id]["shell"] == "EO SSO" else (satellite_b_id, satellite_a_id)
        )
        last_inter_shell_removal[eo_id] = (event.sim_time, satellites[relay_id]["name"])


async def on_gsl_created(event: GSLCreatedEvent):
    open_link(event.link_id, event.ground_station_id, event.satellite_id, "ground", event.sim_time)


async def on_gsl_removed(event: GSLRemovedEvent):
    close_link(event.link_id, event.sim_time)


event_bus = EventBus()
event_bus.subscribe(EventType.ISL_CREATED, on_isl_created)
event_bus.subscribe(EventType.ISL_UPDATED, on_isl_updated)
event_bus.subscribe(EventType.ISL_REMOVED, on_isl_removed)
event_bus.subscribe(EventType.GSL_CREATED, on_gsl_created)
event_bus.subscribe(EventType.GSL_REMOVED, on_gsl_removed)

## 4. A graph of the network, maintained in real time
[`dynamic_topology_graph.py`](dynamic_topology_graph.py) (extracted from the `sat_com_dynamic_graph` package of the satgonetem test environment) keeps an [igraph](https://python.igraph.org) graph in sync with the simulation, from the events only:
- `TO_CREATED` adds a vertex (one per satellite and ground station), named after its `topology_uniq_id`
- `ISL_CREATED` / `GSL_CREATED` add an edge carrying the link `distance_km`, `*_UPDATED` refreshes it, `*_REMOVED` deletes it
- an ISL flagged `temporary_unavailable` leaves the graph until it comes back

`attach(event_bus)` subscribes it next to our own handlers: the same events feed both. Attach it **before** loading the simulation, so it sees the whole initial topology.

At any time, `propagation_latencies_ms(sources, targets)` runs a shortest path weighted by the distance (Dijkstra) and converts it to a propagation delay at the speed of light: `latency (ms) = distance (km) / 299 792.458 × 1000`. Only propagation: no processing, queuing or transmission delay.

In [ ]:
topology_graph = DynamicTopologyGraph().attach(event_bus)

## 5. Load the simulation
As in the optical lab, `run_synchronously` runs the simulation outside Jupyter's event loop, so each handler runs as soon as its event is published (see the note in [optical_terminal_lab.ipynb](optical_terminal_lab.ipynb)).

In [ ]:
def run_synchronously(function, *args, **kwargs):
    """Run `function` outside Jupyter's event loop, so the event bus calls each handler right away."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(function, *args, **kwargs).result()


configuration_manager = BaseConfigurationManager(
    simulation_property=simulation_properties,
    event_bus=event_bus,
    event_factories=event_factories,
)
simulation_manager = run_synchronously(configuration_manager.load_simulation)

ground_stations = {ground_station.label: ground_station for ground_station in simulation_manager.get_ground_stations()}
eo_satellite_ids = sorted(satellite.topology_uniq_id for satellite in simulation_manager.get_sso_shell_satellites())

inter_shell_links = [link for link in open_links.values() if link[2] == "inter_shell"]
print(f"{len(simulation_manager.get_walker_shell_satellites())} EO Relay satellites, {len(eo_satellite_ids)} EO satellites")
print(f"ground stations: {', '.join(f'{label} (id {gs.topology_uniq_id})' for label, gs in ground_stations.items())}")
print(f"{len(open_links)} links at start, {len(inter_shell_links)} of them inter-shell")
print(f"graph: {topology_graph.graph.vcount()} vertices, {topology_graph.graph.ecount()} edges")

## 6. Measure the latencies at every tick
`measure_latencies` is registered on the time manager: it runs at every tick, **after** the simulation update, so the graph has already received every event of the tick. It asks the graph for the latency from each ground station to every EO satellite, and keeps:
- the latency to **every** EO satellite (shown in the Cesium label of each EO satellite)
- the **shortest** one: the nearest EO satellite of each ground station (shown in the Cesium legend)

`inf` means no path: a **blackout**. For CNES it is expected: it only talks to the EO satellites directly, and the 6 of them share one orbit. Around dawn they fly over Toulouse one after the other, about every 16 minutes, each one visible for a few minutes: CNES alternates between a link and a blackout. Once connected to an EO satellite, CNES can also reach the 5 others through the inter-shell links and EO Relay. ISAE-SUPAERO reaches them through EO Relay, so it can talk to any EO satellite that has an inter-shell link.

In [ ]:
latency_history = []
GROUND_STATION_LABELS = ["CNES", "ISAE-SUPAERO"]


def format_latency(latency_ms):
    return "blackout" if math.isinf(latency_ms) else f"{latency_ms:.2f} ms"


def measure_latencies():
    sim_time = simulation_manager.time_manager.current_time
    sources = [ground_stations[label].topology_uniq_id for label in GROUND_STATION_LABELS]
    matrix = topology_graph.propagation_latencies_ms(sources, eo_satellite_ids)

    snapshot = {"time": sim_time, "latencies": {}, "nearest": {}}
    for label, row in zip(GROUND_STATION_LABELS, matrix):
        snapshot["latencies"][label] = dict(zip(eo_satellite_ids, row))
        nearest_latency, nearest_id = min(zip(row, eo_satellite_ids))
        snapshot["nearest"][label] = (nearest_latency, nearest_id)
    latency_history.append(snapshot)

    if sim_time.minute % 10 == 0 and sim_time.second == 0:
        print(f"[{sim_time:%H:%M:%S}] " + " | ".join(
            f"{label} -> {format_latency(latency)}"
            + (f" ({satellites[eo_id]['name']})" if not math.isinf(latency) else "")
            for label, (latency, eo_id) in snapshot["nearest"].items()
        ) + f" | {handover_counter['total']} handovers")


measure_latencies()  # state right after the load
simulation_manager.time_manager.register_action(measure_latencies)

## 7. Run the simulation
Every handover is printed with the running counter, and every 10 minutes the nearest EO satellite of each ground station.

In [ ]:
run_synchronously(simulation_manager.time_manager.tick_until_the_end, ticking_time_in_seconds=30)

end_time = simulation_manager.time_manager.current_time
for link_id in list(open_links):
    close_link(link_id, end_time)

print(f"\n{handover_counter['total']} inter-shell handovers in 2 hours")

## 8. Generate the animated Cesium page
[`cesium_graph_visualizer.py`](cesium_graph_visualizer.py) draws a networkX graph in Cesium. Its `AnimatedCesiumGraphVisualizer` takes the same graph, with extra attributes:
- node `positions`: the list of `(time, longitude, latitude, altitude)` samples. Cesium interpolates between them. A `static` node (our ground stations) only needs one.
- node `label_samples`: the label of the node over time. Each EO satellite shows its latency from both ground stations.
- edge `intervals`: the list of `(start, end)` during which the link exists. Cesium only shows it during these intervals.

and `live_values`, values shown in the legend and updated as the clock runs: the nearest EO satellite of each ground station, and the handover counter.

Without a [Cesium Ion token](https://ion.cesium.com/tokens?page=1), the page uses the offline Natural Earth imagery shipped with CesiumJS.

> **Why not update Cesium in real time?** The graph and the latencies are maintained in real time, but the page is a static HTML file: to update it live, the simulation would have to push its events to the browser through a server (a WebSocket for instance). Here the simulation runs faster than real time anyway, so we run it first and replay it: everything is time-tagged (`sim_time`), which is all Cesium's clock needs.

In [ ]:
CESIUM_TOKEN = None  # or "<YOUR_CESIUM_TOKEN>"


def nearest_text(latency_ms, eo_id):
    return "blackout" if math.isinf(latency_ms) else f"{latency_ms:.2f} ms → {satellites[eo_id]['name']}"


graph = nx.Graph()
for satellite_id, satellite in satellites.items():
    graph.add_node(
        satellite_id,
        type="Satellite",
        name=satellite["name"],
        shell=satellite["shell"],
        positions=[(time, *position) for time, position in satellite["positions"].items()],
    )

# Ground stations do not move: one position is enough. They are 2 km apart: move their labels apart.
label_offsets = {"CNES": (-45, -16), "ISAE-SUPAERO": (60, 16)}
for label, ground_station in ground_stations.items():
    graph.add_node(
        ground_station.topology_uniq_id,
        type="GroundStation",
        name=label,
        shell="Ground Station",
        static=True,
        label_offset=label_offsets[label],
        positions=[(simulation_manager.time_manager.start_date, *position_of(ground_station))],
    )

# Each EO satellite shows its latency from both ground stations.
for eo_id in eo_satellite_ids:
    graph.nodes[eo_id]["label_samples"] = [
        (
            snapshot["time"],
            f"{satellites[eo_id]['name']}\n"
            + "\n".join(f"{label}: {format_latency(snapshot['latencies'][label][eo_id])}" for label in GROUND_STATION_LABELS),
        )
        for snapshot in latency_history
    ]

for (object_a_id, object_b_id), lifetime in link_lifetimes.items():
    graph.add_edge(
        object_a_id,
        object_b_id,
        direction=lifetime["direction"],
        intervals=lifetime["intervals"],
        established_count=len(lifetime["intervals"]),
    )

live_values = {
    f"{label} → nearest EO satellite": [
        (snapshot["time"], nearest_text(*snapshot["nearest"][label])) for snapshot in latency_history
    ]
    for label in GROUND_STATION_LABELS
}
live_values["Inter-shell handovers"] = [(simulation_manager.time_manager.start_date, 0)] + handover_counter["samples"]

print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} pairs linked at least once")

visualizer = AnimatedCesiumGraphVisualizer(
    graph,
    start_time=simulation_manager.time_manager.start_date,
    end_time=end_time,
    title="EO Relay + EO SSO: inter-shell links and latency from Toulouse",
    cesium_token=CESIUM_TOKEN,
    live_values=live_values,
)
visualizer.generate_html("results/inter_shell_animation.html")

Open [`results/inter_shell_animation.html`](results/inter_shell_animation.html) in your browser and zoom on Europe. Things to watch:
- the 6 EO satellites (one orbit, close to the poles, no link between them) cross the EO Relay shell; each one stays connected to one relay satellite (pink link) until the link is no longer feasible, then hands over: the **handover counter** in the legend goes up
- the yellow ground links from Toulouse: ISAE-SUPAERO always has a relay satellite overhead, CNES only gets a link when an EO satellite passes (and shows `blackout` in between)
- the latency labels on the EO satellites and the **nearest EO satellite** of each ground station in the legend

## 9. Recap

In [ ]:
from collections import Counter

handovers_per_eo_satellite = Counter(handover["eo"] for handover in handovers)
for name in sorted(handovers_per_eo_satellite):
    print(f"{name}: {handovers_per_eo_satellite[name]} handovers")

durations = [
    (end - start).total_seconds() / 60
    for lifetime in link_lifetimes.values()
    if lifetime["direction"] == "inter_shell"
    for start, end in lifetime["intervals"]
]
print(f"inter-shell links: {len(durations)}, mean duration {sum(durations) / len(durations):.1f} min, longest {max(durations):.1f} min\n")

for label in GROUND_STATION_LABELS:
    nearest = [snapshot["nearest"][label][0] for snapshot in latency_history]
    reachable = [latency for latency in nearest if not math.isinf(latency)]
    blackout_share = 100 * (len(nearest) - len(reachable)) / len(nearest)
    summary = (
        f"min {min(reachable):.2f} ms, mean {sum(reachable) / len(reachable):.2f} ms, max {max(reachable):.2f} ms"
        if reachable else "never reachable"
    )
    print(f"{label:<13} nearest EO satellite: {summary}, blackout {blackout_share:.0f}% of the ticks")

## What's next?
- Let CNES also use the relay (`shells_whitelist: ["EO SSO", "EO Relay"]`): do its blackouts disappear?
- Give the EO Relay satellites **two** inter-shell terminals, or the EO satellites two: how do the handovers and the ISAE-SUPAERO latency change?
- Move the EO orbit up or down (`altitude_in_km`) and compare the inter-shell link durations.
- Use `topology_graph.hop_counts(...)` to compare the number of hops with the latency.
- Send the events to a real emulator: the library README, *How to use satComTopology in my emulator*, shows how to inject your own satellite classes and events.